# TiGeSR · GIRAFFE

На GPU ещё не проверен.

Запускайте сверху вниз в отдельной сессии Kaggle с GPU и Internet. Зависимости устанавливаются в отдельное Python-окружение.

Подключите датасет через Add Input и укажите его путь в разделе 4. Структура: `<корень>/<sample_id>/degraded.png`; `hint.png` нужен только для просмотра.

[Официальный репозиторий](https://github.com/OpenVeraTeam/TiGeSR) · [Зафиксированная версия](https://github.com/OpenVeraTeam/TiGeSR/tree/136cb65444868d80ffa28beb37919a94564f3b5b)

## 1. Репозиторий


In [ ]:
from pathlib import Path
import datetime
import hashlib
import json
import os
import shutil
import struct
import subprocess
import sys
import uuid
import zipfile

REPO_URL = 'https://github.com/OpenVeraTeam/TiGeSR.git'
COMMIT = '136cb65444868d80ffa28beb37919a94564f3b5b'
WORK_ROOT = Path('/kaggle/working/giraffe_baselines') if Path('/kaggle/working').exists() else Path.cwd() / '.baseline_work'
WORK_ROOT = WORK_ROOT.resolve()
REPO_DIR = WORK_ROOT / 'external' / 'tigesr'

def run(*args, cwd=REPO_DIR, env=None):
    subprocess.run([str(a) for a in args], cwd=cwd, env=env, check=True)

REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    run('git', 'clone', REPO_URL, REPO_DIR, cwd=WORK_ROOT)
origin = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
if origin.removesuffix('.git') != REPO_URL.removesuffix('.git'):
    raise RuntimeError(f'В {REPO_DIR} находится другой репозиторий: {origin}')
if subprocess.check_output(['git', '-C', str(REPO_DIR), 'diff', 'HEAD', '--name-only'], text=True).strip():
    raise RuntimeError('В baseline-репозитории изменены отслеживаемые файлы. Сохраните изменения отдельно.')
run('git', 'fetch', 'origin', COMMIT)
run('git', 'checkout', '--detach', COMMIT)
actual = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if actual != COMMIT:
    raise RuntimeError(f'Версия репозитория не совпала: {actual}')
print(f'tigesr: {actual}')


## 2. Окружение

Используется комплект UZ-ST. PP-OCRv5 автоматически читает входные изображения в отдельном CPU-окружении. TiGeSR восстанавливает их с ограничением `max_side=1024`; это не увеличение ×4. Текстовое условие — официальный `writing_nocfg` cache.


In [ ]:
# Зависимости метода устанавливаются отдельно от ядра Jupyter.
if sys.platform != 'linux' or not shutil.which('nvidia-smi'):
    raise RuntimeError('Нужна Linux-сессия с NVIDIA GPU. В Kaggle включите GPU и Internet. На Mac CUDA недоступна.')
run('nvidia-smi', cwd=WORK_ROOT)
run(sys.executable, '-m', 'pip', 'install', 'uv==0.9.7', cwd=WORK_ROOT)
ENV_DIR = WORK_ROOT / 'envs' / 'tigesr'
if not (ENV_DIR / 'bin/python').exists():
    run(sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '--seed', ENV_DIR, cwd=WORK_ROOT)
PYTHON = ENV_DIR / 'bin/python'

def run_python(source, *args, env=None):
    run(PYTHON, '-c', source, *args, env=env)

run(PYTHON, '-m', 'pip', 'install', 'pip==24.3.1', 'setuptools==75.6.0', 'wheel==0.45.1')
run(PYTHON, '-m', 'pip', 'install', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu121')
run(PYTHON, '-m', 'pip', 'install', '-r', 'requirements.txt', 'gdown==5.2.0')
OCR_ENV = WORK_ROOT / 'envs' / 'tigesr_ocr'
if not (OCR_ENV / 'bin/python').exists():
    run(sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '--seed', OCR_ENV)
OCR_PYTHON = OCR_ENV / 'bin/python'
run(OCR_PYTHON, '-m', 'pip', 'install', '-r', 'requirements/ocr.txt')
run(OCR_PYTHON, '-m', 'pip', 'check')
run_python("import torch; assert torch.cuda.is_available(), 'PyTorch не видит CUDA'; print(torch.__version__, torch.cuda.get_device_name(0))")
run(PYTHON, '-m', 'pip', 'check')


## 3. Веса

Веса UZ-ST, Hi-SAM и prompt cache скачиваются с Hugging Face, DiffTSR — с Google Drive. Для SD3.5 Medium нужно принять условия на Hugging Face и добавить токен чтения `HF_TOKEN` через Kaggle **Add-ons → Secrets** и включить доступ для ноутбука. Ячейка передаст его загрузчику через окружение. Если Hi-SAM потребует `sam_vit_h_4b8939.pth`, загрузите его в `weights/` по инструкции авторов. При ограничении Google Drive используйте путь из сообщения об ошибке.

Загрузка начинается при запуске ячейки. Уже скачанные файлы используются повторно.


In [ ]:
if not os.environ.get('HF_TOKEN'):
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('Добавьте HF_TOKEN в Kaggle Add-ons → Secrets и разрешите этому ноутбуку доступ. Аккаунту токена нужен доступ к stabilityai/stable-diffusion-3.5-medium.') from exc

run_python(r"""from pathlib import Path
import json, shutil, hashlib
from huggingface_hub import hf_hub_download, snapshot_download, model_info
import gdown

# В /kaggle/working только 20 ГБ: веса храним на временном диске сессии.
weights = Path('weights')
weight_store = Path('/tmp/giraffe_tigesr_weights')
if not weights.is_symlink():
    if weights.exists():
        if weight_store.exists():
            raise RuntimeError(f'{weight_store} уже существует; проверьте незавершённый перенос весов.')
        shutil.move(str(weights), str(weight_store))
    else:
        weight_store.mkdir(parents=True, exist_ok=True)
    weights.symlink_to(weight_store, target_is_directory=True)
print(f'Веса: {weights.resolve()}; свободно {shutil.disk_usage(weight_store).free / 2**30:.1f} ГиБ')
assets = []

def hf_file(repo, filename, destination, revision):
    destination = Path(destination)
    if not destination.is_file() or destination.stat().st_size == 0:
        src = hf_hub_download(repo_id=repo, filename=filename, revision=revision)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.symlink_to(Path(src).resolve())
    assets.append({'source': f'https://huggingface.co/{repo}', 'revision': revision,
                   'file': filename, 'local': str(destination), 'bytes': destination.stat().st_size})
    return destination

def hf_snapshot(repo, destination, patterns, revision=None):
    destination = Path(destination)
    revision = revision or model_info(repo).sha
    snapshot_download(repo_id=repo, revision=revision, local_dir=destination,
                      allow_patterns=patterns, ignore_patterns=['*.onnx*', '*.msgpack', '*.fp16.*', '*.pyc'])
    assets.append({'source': f'https://huggingface.co/{repo}', 'revision': revision,
                   'local': str(destination), 'patterns': patterns})
    return destination

def drive_file(folder_url, filename, destination):
    destination = Path(destination)
    if not destination.is_file() or destination.stat().st_size == 0:
        entries = gdown.download_folder(url=folder_url, skip_download=True, quiet=True)
        matches = [entry for entry in (entries or []) if Path(entry.path).name == filename]
        if len(matches) != 1:
            raise RuntimeError(f'Не найден однозначный {filename}. Скачайте его вручную из {folder_url} и положите в {destination}; затем повторите ячейку.')
        destination.parent.mkdir(parents=True, exist_ok=True)
        part = destination.with_name(destination.name + '.part')
        if not gdown.download(id=matches[0].id, output=str(part), quiet=False, resume=True):
            raise RuntimeError(f'Google Drive не отдал {filename}. Загрузите файл вручную в {destination}.')
        part.replace(destination)
    with destination.open('rb') as f:
        digest = hashlib.file_digest(f, 'sha256').hexdigest() if hasattr(hashlib, 'file_digest') else None
    if digest is None:
        h = hashlib.sha256()
        with destination.open('rb') as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
                h.update(chunk)
        digest = h.hexdigest()
    assets.append({'source': folder_url, 'file': filename, 'local': str(destination),
                   'bytes': destination.stat().st_size, 'sha256': digest})
    return destination

revision = 'f473ac373da2b95b84a3ed3c0b72b0944d919972'
for suffix in ['stage1/model.pth', 'stage1/encoder.pth', 'stage2/model.pkl']:
    hf_file('mxluocv/TiGeSR', 'uzst/' + suffix, 'weights/uzst/' + suffix, revision)
for remote, local in {
    'SAM/BTS-test-test_best.pth': 'weights/BTS-test-test_best.pth',
    'PNDMscheduler/scheduler_config.json': 'weights/DiffTSR/PNDMscheduler/scheduler_config.json',

}.items(): hf_file('mxluocv/TiGeSR', remote, local, revision)
drive_file('https://drive.google.com/drive/folders/1K6k5ZcvF3w-1MDN_gXQTdsLgFZ2SM8qy',
           'DiffTSR.ckpt', 'weights/DiffTSR/DiffTSR_new.ckpt')
hf_snapshot('stabilityai/stable-diffusion-3.5-medium', 'weights/stable-diffusion-3.5-medium',
            ['model_index.json', 'transformer/*', 'vae/*', 'scheduler/*'])
hf_file('mxluocv/TiGeSR',
    'weights/Stable-Diffusion-3.5-medium/saved_prompt_tokens_writing_nocfg.pt',
    'weights/stable-diffusion-3.5-medium/saved_prompt_tokens_writing_nocfg.pt', revision)
Path('giraffe_assets.json').write_text(json.dumps({'assets': assets}, indent=2))
""")


## 4. Параметры и данные

Вход модели — только `degraded.png`. Каждое выполнение создаёт новую папку запуска.


In [ ]:
DATASET_ROOT = Path('/kaggle/input/datasets/markunyaa/giraffe-dataset/giraffe_dataset')
RESULTS_ROOT = WORK_ROOT / 'results'
SEED = 42
SETTINGS = {
    'checkpoint_set': 'uzst',
    'max_side': 1024,
    'timestep': 150,
    'ctrl_layers': 6,
    'enable_tiling': True,
    'prompt_cache': 'saved_prompt_tokens_writing_nocfg.pt',
}

DATASET_ROOT = DATASET_ROOT.expanduser().resolve()
RESULTS_ROOT = RESULTS_ROOT.expanduser().resolve()
sources = sorted(DATASET_ROOT.glob('*/degraded.png'))
if not sources:
    raise FileNotFoundError(f'Не найдены */degraded.png в {DATASET_ROOT}. Укажите путь из Kaggle Add Input.')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = WORK_ROOT / 'runs' / 'tigesr' / RUN_ID
INPUT_DIR = RUN_DIR / 'input'
RAW_DIR = RUN_DIR / 'raw'
INPUT_DIR.mkdir(parents=True)
RAW_DIR.mkdir()
SAMPLES = {}
for src in sources:
    sample_id = src.parent.name
    target = INPUT_DIR / f'{sample_id}.png'
    shutil.copy2(src, target)
    SAMPLES[sample_id] = src
ASSETS = json.loads((REPO_DIR / 'giraffe_assets.json').read_text())
RUN_CONFIG = {'method': 'tigesr', 'repo': REPO_URL, 'commit': COMMIT, 'seed': SEED,
              'settings': SETTINGS, 'weights': ASSETS, 'dataset_root': str(DATASET_ROOT),
              'input_dir': str(INPUT_DIR), 'raw_dir': str(RAW_DIR), 'run_id': RUN_ID}
CONFIG_PATH = RUN_DIR / 'run.json'
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))
INFERENCE_OK = False
EXPORT_OK = False
print(f'{len(SAMPLES)} примеров: ' + ', '.join(SAMPLES))
print(f'Результаты: {RESULTS_ROOT}')


## 5. Инференс

Для повторного запуска сначала выполните ячейку параметров.


In [ ]:
INFERENCE_OK = False
EXPORT_OK = False
if any(RAW_DIR.iterdir()):
    raise RuntimeError('Для повторного инференса заново выполните ячейку параметров: она создаст новую папку запуска.')

# FP16 только для SD3 Transformer и ControlNet; VAE и первая стадия остаются FP32.
source = (REPO_DIR / 'scripts/inference.py').read_text()
anchor = '    net_sr.set_eval()'
replacement = """    for module in (net_sr.net_pix2pix.transformer, net_sr.net_pix2pix.controlnet):
        module.half()
        module.forward = torch.autocast('cuda', dtype=torch.float16)(module.forward)
    net_sr.set_eval()"""
if source.count(anchor) != 1:
    raise RuntimeError('Изменился код официального инференса: проверьте FP16-адаптацию.')
source = source.replace(anchor, replacement)
source = source.replace('        save_image(sr_img.cpu(), output_path)',
    '        if not torch.isfinite(sr_img).all():\n'
    '            raise RuntimeError(f"Non-finite output: {stem}")\n'
    '        print(f"Saving {stem}; GPU peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB", flush=True)\n'
    '        save_image(sr_img.cpu(), output_path)')
source = source.replace('    net_sr.to_device(torch.device(f"cuda:{rank}"))', '''    device = torch.device(f"cuda:{rank}")
    net_sr.execution_device = device

    def activate_stage2(module, inputs):
        if module.execution_device != device:
            net_sr.net_mask_gen.to_device(torch.device("cpu"))
            torch.cuda.empty_cache()
            module.to_device(device)
            print("Stage 2: SD3 on GPU", flush=True)

    def load_denoiser(module, inputs):
        module.to(device)

    def unload_denoiser(module, inputs, output):
        module.to("cpu")
        torch.cuda.empty_cache()

    for module in (net_sr.net_pix2pix.transformer, net_sr.net_pix2pix.controlnet):
        module.register_forward_pre_hook(load_denoiser)
        module.register_forward_hook(unload_denoiser)

    net_sr.net_pix2pix.register_forward_pre_hook(activate_stage2)''')
source = source.replace('        with torch.no_grad():', '''        net_sr.net_pix2pix.to_device(torch.device("cpu"))
        torch.cuda.empty_cache()
        net_sr.net_mask_gen.to_device(device)
        print(f"Stage 1: {stem}", flush=True)
        with torch.no_grad():''')
INFERENCE_SCRIPT = REPO_DIR / 'scripts/giraffe_inference_fp16.py'
INFERENCE_SCRIPT.write_text(source)
(RUN_DIR / 'inference_fp16.py').write_text(source)
RUN_CONFIG['precision'] = {'sd3_transformer': 'fp16_autocast', 'controlnet': 'fp16_autocast',
                           'vae': 'fp32', 'stage1': 'fp32', 'offload': 'stagewise_and_denoiser_cpu'}
RUN_CONFIG['inference_script_sha256'] = hashlib.sha256(source.encode()).hexdigest()
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))

ANNOTATIONS = RUN_DIR / 'ocr'
OCR_COMMAND = [str(OCR_PYTHON), 'scripts/make_ocr_annotations.py', '--input-dir', str(INPUT_DIR),
    '--output-dir', str(ANNOTATIONS), '--device', 'cpu', '--max-side', str(SETTINGS['max_side'])]
run(*OCR_COMMAND)
COMMAND = [str(PYTHON), str(INFERENCE_SCRIPT), '--test_folder', str(INPUT_DIR),
    '--anno_path', str(ANNOTATIONS), '--output_dir', str(RAW_DIR),
    '--difftsr_config', 'configs/stage1.yaml',
    '--idm_ckpt_path', 'weights/uzst/stage1/model.pth', '--enc_path', 'weights/uzst/stage1/encoder.pth',
    '--ctrlnet_ckpt_path', 'weights/uzst/stage2/model.pkl',
    '--prompt_cache_path', 'weights/stable-diffusion-3.5-medium/' + SETTINGS['prompt_cache'],
    '--timestep', str(SETTINGS['timestep']), '--ctrl_layers', str(SETTINGS['ctrl_layers']),
    '--max_side', str(SETTINGS['max_side']), '--seed', str(SEED)]
if SETTINGS['enable_tiling']:
    COMMAND.append('--enable_tiling')
RUN_ENV = os.environ.copy()
RUN_ENV['CUDA_VISIBLE_DEVICES'] = '0'
RUN_ENV['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
RUN_ENV['PYTHONUNBUFFERED'] = '1'
run(*COMMAND, env=RUN_ENV)
PREDICTIONS = {sid: RAW_DIR / f'{sid}.png' for sid in SAMPLES}
INFERENCE_OK = True


## 6. Сохранение

Результаты: `results/<sample_id>/tigesr.png`. Рядом сохраняются параметры, версии весов, seed и размеры изображений. Архив содержит результаты и сведения о запуске. Повторный экспорт заменяет PNG и JSON этого метода; архивы сохраняются отдельно.


In [ ]:
EXPORT_OK = False
if not INFERENCE_OK:
    raise RuntimeError('Сначала выполните инференс текущего запуска без ошибок.')
missing = [str(p) for p in PREDICTIONS.values() if not p.is_file()]
if missing:
    raise RuntimeError('Метод не создал все результаты: ' + ', '.join(missing))

def png_size(path):
    with Path(path).open('rb') as f:
        header = f.read(24)
    if header[:8] != b'\x89PNG\r\n\x1a\n':
        raise ValueError(f'Ожидался PNG: {path}')
    return list(struct.unpack('>II', header[16:24]))

for path in PREDICTIONS.values():
    png_size(path)
freeze = subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], text=True)
(RUN_DIR / 'environment.txt').write_text(freeze)
RUN_CONFIG['command'] = COMMAND
RUN_CONFIG['status'] = 'completed'
RUN_CONFIG['completed_at_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
RUN_CONFIG['ocr_command'] = OCR_COMMAND
RUN_CONFIG['cuda_visible_devices'] = '0'
(RUN_DIR / 'ocr_environment.txt').write_text(subprocess.check_output([str(OCR_PYTHON), '-m', 'pip', 'freeze'], text=True))
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))
EXPORTED = []
for sid, predicted in PREDICTIONS.items():
    out_dir = RESULTS_ROOT / sid
    out_dir.mkdir(parents=True, exist_ok=True)
    output = out_dir / 'tigesr.png'
    shutil.copy2(predicted, output)
    metadata = dict(RUN_CONFIG, sample_id=sid,
        input_size=png_size(INPUT_DIR / f'{sid}.png'), output_size=png_size(output),
        input_sha256=hashlib.sha256((INPUT_DIR / f'{sid}.png').read_bytes()).hexdigest(),
        output_sha256=hashlib.sha256(output.read_bytes()).hexdigest())
    info = out_dir / 'tigesr.json'
    info.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
    EXPORTED.extend([output, info])
archive = WORK_ROOT / f'giraffe_tigesr_{RUN_ID}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in EXPORTED:
        z.write(path, Path('results') / path.relative_to(RESULTS_ROOT))
    z.write(CONFIG_PATH, 'run.json')
    z.write(RUN_DIR / 'inference_fp16.py', 'inference_fp16.py')
    z.write(RUN_DIR / 'environment.txt', 'environment.txt')
    z.write(RUN_DIR / 'ocr_environment.txt', 'ocr_environment.txt')
    for path in ANNOTATIONS.glob('*.json'):
        z.write(path, Path('ocr') / path.name)
EXPORT_OK = True
print(f'Сохранено {len(PREDICTIONS)} результатов: {archive}')


## 7. Просмотр

`hint.png` — референс для сравнения. Он не поступает в модель и может отличаться ракурсом или кадрированием.


In [ ]:
if not EXPORT_OK:
    raise RuntimeError('Сначала завершите сохранение результатов текущего запуска.')
from IPython.display import display, HTML, FileLink
import base64
import html
for sid, source in SAMPLES.items():
    items = [('Вход: degraded.png', source), ('Результат: tigesr', RESULTS_ROOT / sid / 'tigesr.png')]
    hint = source.parent / 'hint.png'
    if hint.is_file():
        items.append(('Референс (hint.png)', hint))
    panels = []
    for label, path in items:
        encoded = base64.b64encode(path.read_bytes()).decode('ascii')
        panels.append(f'<div style="max-width:32%;"><p>{html.escape(label)}</p><img style="width:100%;height:auto" src="data:image/png;base64,{encoded}"></div>')
    display(HTML('<h3>' + html.escape(sid) + '</h3><div style="display:flex;gap:12px;align-items:flex-start">' + ''.join(panels) + '</div>'))
# Относительная ссылка обслуживается Jupyter; в Kaggle архив также виден в Output.
display(FileLink(os.path.relpath(archive, Path.cwd())))
